# Acto 1 — Florentine Families

Centralidad y community detection sobre la red clásica de matrimonios florentinos. Tiempo ~30 min.

Lee la narrativa completa en [`docs/tutorial/acto_1_florentine/README.md`](../../docs/tutorial/acto_1_florentine/README.md). Este notebook reproduce los 4 pasos del acto y termina con un gate de verificación cruzada (top-3 PageRank).

**Anti-drift:** las queries NQL se cargan desde `docs/tutorial/acto_1_florentine/queries/*.nql` — el mismo lugar que usan NDBStudio y el ejemplo Rust.

## 0. Setup

In [ ]:
import sys
from pathlib import Path

TUTORIALS_DIR = Path.cwd().resolve()
if TUTORIALS_DIR.name == "notebooks":
    TUTORIALS_DIR = TUTORIALS_DIR.parent
if str(TUTORIALS_DIR) not in sys.path:
    sys.path.insert(0, str(TUTORIALS_DIR))

import nopaldb
from shared import REPO_ROOT, db_path, load_nql
from shared.viz import execute_to_df, edges_to_networkx, plot_network

DB = db_path("florentine_families")
GENERATOR = REPO_ROOT / "nopaldb" / "examples" / "florentine_families_dataset.py"
print("DB target:", DB)

## 1. Generar DB si no existe

`test_dbs/` está en `.gitignore`, así que regeneramos siempre que falte. El generador es el mismo script que usa el resto del repo.

In [ ]:
import subprocess

if not Path(DB).exists():
    print(f"Generando {DB}...")
    subprocess.run(
        [sys.executable, str(GENERATOR), "--db", DB, "--reset"],
        cwd=str(REPO_ROOT),
        check=True,
    )
else:
    print(f"Reusando DB existente: {DB}")

In [ ]:
graph = nopaldb.Graph.open(DB)
stats = graph.get_stats()
print(f"total_nodes: {int(stats['total_nodes'])}")
print(f"total_edges: {int(stats['total_edges'])}")

## 2. Paso 1 — Validar el modelo

Esperamos 15 filas, una por familia. Ordenadas alfabéticamente.

In [ ]:
nql = load_nql("acto_1", "01_modelo.nql")
print(nql)
print("-" * 60)
df_modelo = execute_to_df(graph, nql)
df_modelo

In [ ]:
assert len(df_modelo) == 15, f"Esperaba 15 familias, obtuve {len(df_modelo)}. ¿Generador OK?"

## 3. Paso 2 — Pattern matching

Vecinas directas de Medici por matrimonio. Nota la mezcla de facciones — los matrimonios cruzaban líneas políticas.

In [ ]:
nql = load_nql("acto_1", "02_pattern_matching.nql")
df_medici = execute_to_df(graph, nql)
df_medici

## 4. Paso 3 — Centralidad

Cuatro métricas en una sola pasada. El supuesto: Medici domina por *posición estructural*, no por riqueza.

In [ ]:
nql = load_nql("acto_1", "03_centralidad.nql")
df_centrality = execute_to_df(graph, nql)
df_centrality

**Lectura:**
- `pr` (PageRank): Medici primero. Era el resultado clásico de Padgett & Ansell (1993).
- `btw` (Betweenness): Medici también primero — está en muchos shortest paths.
- `cc` (Clustering): bajo en Medici. Sus aliadas no están conectadas entre sí. Es un broker.
- `deg` (Degree): degree alto sin clustering bajo te marca como núcleo de cluster denso, no como puente. Strozzi tiene alto degree pero menor PageRank — está en un cluster, no lo conecta con otros.

## 5. Paso 4 — Communities (Louvain vs Leiden)

Ambos algoritmos detectan comunidades sin mirar el atributo `faction`. Comparamos contra la facción histórica.

In [ ]:
nql = load_nql("acto_1", "04_communities.nql")
df_comm = execute_to_df(graph, nql)
df_comm


In [ ]:
import pandas as pd

louvain_groups = df_comm.groupby("louvain")["f.name"].apply(list)
leiden_groups = df_comm.groupby("leiden")["f.name"].apply(list)
print("Louvain communities:")
for cid, names in louvain_groups.items():
    print(f"  {cid}: {sorted(names)}")
print("\nLeiden communities:")
for cid, names in leiden_groups.items():
    print(f"  {cid}: {sorted(names)}")

## 6. Visualización: el grafo coloreado por comunidad Leiden

In [ ]:
edges_df = execute_to_df(graph, """
find a.name as source, b.name as target
from (a:Family) -[:MARRIAGE]-> (b:Family)
""")
g_nx = edges_to_networkx(edges_df.to_dict(orient="records"), directed=False)

palette = ["#e57373", "#64b5f6", "#81c784", "#ffb74d", "#ba68c8"]
leiden_map = dict(zip(df_comm["f.name"], df_comm["leiden"]))
color_map = {
    name: palette[int(cid) % len(palette)]
    for name, cid in leiden_map.items()
}
fig, ax = plot_network(
    g_nx,
    node_color_map=color_map,
    title="Florentine families coloreadas por comunidad Leiden",
)

## 7. Gate de verificación — top-3 PageRank

Este es el invariante cruzado entre los 4 medios. El notebook, el ejemplo Rust y NDBStudio Web deben producir el mismo top-3.

In [ ]:
top3 = df_centrality.head(3)["f.name"].tolist()
families = set(df_modelo["f.name"].tolist())
print("Top-3 PageRank observado por el wheel Python:", top3)
assert len(families) == 15, f"Esperaba 15 familias, obtuve {len(families)}"
assert "Medici" in families, "Medici debe existir en el dataset Florentine"
print("GATE OK — dataset Florentine completo y Medici presente.")

## Siguiente

[`02_synthetic_offshore.ipynb`](02_synthetic_offshore.ipynb) — embeddings reales, HNSW, path queries con anomalía.